# 06 — 已知时序概率模型：贝叶斯滤波

## 1. 问题定义：目标发生了变化

前 01–05 章的主线是从观测数据学习模型参数，最大化 $p_\theta(y)$（或其下界）。本章暂时固定模型参数，不做学习，专门研究状态推断。

$x_t$ 是不可直接观测的真实状态，$y_t$ 是可以获得的观测。已知：

- 初始状态分布 $p(x_0)$；
- 状态转移模型 $p(x_t\mid x_{t-1})$；
- 观测模型 $p(y_t\mid x_t)$；
- 截至当前的观测 $y_{0:t}=(y_0,\ldots,y_t)$。

假设给定上一状态后，当前状态与更早的状态及观测条件独立；给定当前状态后，当前观测与其余历史条件独立。这些是模型假设，不是普遍成立的概率恒等式。

目标：计算 $p(x_t\mid y_{0:t})$，不是求模型参数，也不是直接求一个状态值。需要单个状态估计时，可以取后验概率最大的类别。

## 2. 从两个时刻推导，而不是记忆公式

先用一个完全离散的例子：$x_t,y_t\in\{0,1\}$。初始状态概率均为 0.5，转移矩阵为

$$
A=\begin{pmatrix}0.9&0.1\\0.1&0.9\end{pmatrix},
$$

观测条件概率矩阵（行是状态，列是观测）为

$$
B=\begin{pmatrix}0.8&0.2\\0.2&0.8\end{pmatrix}.
$$

获得 $y_0=1$ 后，普通贝叶斯公式给出 $p(x_0=0\mid y_0)=0.2$、$p(x_0=1\mid y_0)=0.8$。

现在获得 $y_1=1$。目标是 $p(x_1\mid y_0,y_1)$。直接应用贝叶斯公式：

$$
p(x_1\mid y_0,y_1)=\frac{p(y_1\mid x_1,y_0)p(x_1\mid y_0)}{p(y_1\mid y_0)}=\frac{p(y_1\mid x_1)p(x_1\mid y_0)}{p(y_1\mid y_0)}.
$$

第一项是已知观测模型；剩下两个未知项需要分别计算。

### 2.1 计算预测分布 $p(x_1\mid y_0)$

引入上一状态，边缘化并展开条件联合概率：

$$
\begin{aligned}
p(x_1\mid y_0)&=\sum_{x_0}p(x_1,x_0\mid y_0)\\
&=\sum_{x_0}p(x_1\mid x_0,y_0)p(x_0\mid y_0)\\
&=\sum_{x_0}p(x_1\mid x_0)p(x_0\mid y_0).
\end{aligned}
$$

最后一步来自状态转移的条件独立假设。右侧两项均已知。

| $x_0$ | $x_1$ | $p(x_0\mid y_0)$ | $p(x_1\mid x_0)$ | $p(x_0,x_1\mid y_0)$ |
|---|---|---|---|---|
| 0 | 0 | 0.2 | 0.9 | 0.18 |
| 0 | 1 | 0.2 | 0.1 | 0.02 |
| 1 | 0 | 0.8 | 0.1 | 0.08 |
| 1 | 1 | 0.8 | 0.9 | 0.72 |

按 $x_1$ 分组，对 $x_0$ 求和，得到预测分布 $(0.26,0.74)$。尚未使用新观测。

### 2.2 计算观测预测概率 $p(y_1\mid y_0)$

同样通过边缘化与条件概率展开：

$$
\begin{aligned}
p(y_1\mid y_0)&=\sum_{x_1}p(y_1,x_1\mid y_0)\\
&=\sum_{x_1}p(y_1\mid x_1,y_0)p(x_1\mid y_0)\\
&=\sum_{x_1}p(y_1\mid x_1)p(x_1\mid y_0).
\end{aligned}
$$

| $x_1$ | $y_1$ | $p(x_1\mid y_0)$ | $p(y_1\mid x_1)$ | $p(x_1,y_1\mid y_0)$ |
|---|---|---|---|---|
| 0 | 0 | 0.26 | 0.8 | 0.208 |
| 0 | 1 | 0.26 | 0.2 | 0.052 |
| 1 | 0 | 0.74 | 0.2 | 0.148 |
| 1 | 1 | 0.74 | 0.8 | 0.592 |

按 $y_1$ 分组，对 $x_1$ 求和，得到 $(0.356,0.644)$。实际观测为 1，因此分母为 0.644。

### 2.3 更新后验

$$
p(x_1=0\mid y_0,y_1=1)=0.052/0.644\approx0.081,
$$

$$
p(x_1=1\mid y_0,y_1=1)=0.592/0.644\approx0.919.
$$

本次计算的后验就是下次预测的输入。整个推导只用了条件概率展开、边缘化和贝叶斯公式。

## 3. 推广到任意时刻，与代码对应

预测：

$$
p(x_t\mid y_{0:t-1})=\sum_{x_{t-1}}p(x_t\mid x_{t-1})p(x_{t-1}\mid y_{0:t-1}).
$$

更新：

$$
p(x_t\mid y_{0:t})=\frac{p(y_t\mid x_t)p(x_t\mid y_{0:t-1})}{\sum_k p(y_t\mid x_t=k)p(x_t=k\mid y_{0:t-1})}.
$$

$t=0$ 时直接用初始分布作为预测先验。若隐状态连续，求和改为积分。

下面的代码回到第 02 章的三个高斯簇：状态离散，观测为二维连续变量，因此观测模型返回的是概率密度而非概率。真实状态 x 只用于生成 y 和评估，不进入滤波。

| 数学量 | 代码 |
|---|---|
| $p(x_t=j\mid x_{t-1}=i)$ | A[i,j] |
| $p(y_t\mid x_t=k)$ | models[k].pdf(y[t]) / conditional[t,k] |
| $p(x_t=k\mid y_{0:t-1})$ | predicted[k] |
| $p(x_t=k,y_t\mid y_{0:t-1})$ | joint[k] |
| $p(y_t\mid y_{0:t-1})$ | evidence |
| $p(x_t=k\mid y_{0:t})$ | posterior[k] |

注意 evidence 是观测预测密度，不仅用于归一化，也将成为第 06.5 章极大似然学习的基础。

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
mu = np.array([[-2.,0.],[2.,0.],[0.,3.]])
sigma = 1.1
K = 3
# 转移矩阵 A[i,j] = p(x_t=j | x_{t-1}=i)。
A = np.full((K,K),.025)
np.fill_diagonal(A,.95)
T = 300
# 仅用于仿真产生状态和观测；滤波不使用真实状态。
x = np.empty(T,dtype=int)
x[0] = rng.integers(K)
for t in range(1,T):
    x[t] = rng.choice(K,p=A[x[t-1]])
noise = rng.normal(size=(T,2))
y = np.zeros((T,2))
for t in range(T):
    y[t] = mu[x[t]] + sigma*noise[t]
# 与第 02 章相同：每个类别对应一个条件高斯分布。
models = []
for k in range(K):
    models.append(multivariate_normal(mu[k],sigma**2*np.eye(2)))
conditional = np.zeros((T,K))
for t in range(T):
    for k in range(K):
        conditional[t,k] = models[k].pdf(y[t])

def filter_sequence(conditional,A):
    # 初始状态先验 p(x_0) 为均匀分布。
    posterior = np.ones(K)/K
    filtered = []
    log_evidence = []
    for t in range(len(conditional)):
        # 预测：对上一时刻的状态求和，得到当前状态的预测先验。
        predicted = np.zeros(K)
        if t == 0:
            predicted = posterior.copy()
        else:
            for j in range(K):
                for i in range(K):
                    predicted[j] += A[i,j]*posterior[i]
        # 更新：沿用第 02 章的先验 × 条件密度。
        joint = np.zeros(K)
        for k in range(K):
            joint[k] = conditional[t,k]*predicted[k]
        # 边缘化：当前观测的预测密度 p(y_t | 历史观测)。
        evidence = sum(joint)
        # 贝叶斯归一化：得到当前状态后验。
        for k in range(K):
            posterior[k] = joint[k]/evidence
        # copy 保存当前时刻结果，避免下一时刻原地修改覆盖历史。
        filtered.append(posterior.copy())
        log_evidence.append(np.log(evidence))
    return np.array(filtered),np.array(log_evidence)

# 均匀转移意味着每个时刻独立，退化成静态 GMM。
static, _ = filter_sequence(conditional,np.ones((K,K))/K)
filtered, ll = filter_sequence(conditional,A)
for row in filtered:
    assert np.isclose(sum(row),1)
print('static accuracy:',(static.argmax(1)==x).mean())
print('filter accuracy:',(filtered.argmax(1)==x).mean())
fig,ax = plt.subplots(figsize=(10,3))
ax.plot(x[:100],label='truth')
ax.plot(static.argmax(1)[:100],'.',label='static')
ax.plot(filtered.argmax(1)[:100],'--',label='filter')
ax.legend()
plt.show()

改变观测噪声、转移概率，观察静态后验和滤波后验的差异。第 06.5 章将撤销模型参数已知的假设；第 07 章将状态改为连续线性高斯变量。